# Team Model Job — สมาชิก 07

ไฟล์นี้รับหนึ่งโมเดลต่อหนึ่ง Run All แก้ `OWNER`, `TEAM_SIZE`, `ROUND` ใน Cell 2 แล้วปล่อย `JOB_ID = None` เพื่อรับตามรอบ

ถ้าจะหยิบงานว่างเอง ให้จองเลขกับทีมก่อนแล้วใส่ `JOB_ID = 1..41` ผลจะแยกในโฟลเดอร์ `member-07/job-XX`

ใช้ repository, split กลาง, mode และสูตรฝึกเดียวกันทั้งทีม ดูกติกาเต็มใน `README.md`

In [ ]:
from pathlib import Path
import sys

# ===== แต่ละคนแก้ส่วนนี้ก่อนรับงาน =====
OWNER = "your_name"
TEAM_SIZE = 7  # จำนวนคนที่กำลังช่วยกันรอบนี้: 6 หรือ 7
MEMBER_ID = 7  # หมายเลขของตัวเอง ไม่ซ้ำกัน: 1..TEAM_SIZE
ROUND = 1  # รอบ 1, 2, 3... แต่ละรอบแต่ละคนได้หนึ่งโมเดล
JOB_ID = None  # None = แจกตาม ROUND; หรือใส่เลข 1..41 เพื่อหยิบงานว่างเอง

# ===== ตั้งค่า path เฉพาะเมื่อเครื่องนั้นจำเป็น =====
MODE = "quick"  # ทุกงานในรอบเปรียบเทียบต้องใช้ mode เดียวกัน
REPO_PATH = ""  # Colab: path ของ repository ที่ clone/upload มาทั้งชุด
DATA_PATH = ""  # ว่าง = ใช้ .env THAI_CHAR_DATA_DIR หรือ data/raw
SPLIT_PATH = ""  # ว่าง = data/splits; ทุกคนต้องใช้ split กลางชุดเดียวกัน
OUTPUT_ROOT = ""  # Colab: โฟลเดอร์บน Drive สำหรับเก็บผลถาวร
DOWNLOAD_IF_MISSING = True
DEVICE = "auto"

# ===== สูตรกลาง ห้ามเปลี่ยนรายคน =====
# หากต้องเปลี่ยนค่าเหล่านี้ ให้ตกลงทั้งทีมแล้วเริ่มรอบใหม่พร้อมกัน
SEED = 42
IMAGE_SIZE = 224
BATCH_SIZE = 32
PAD_VALUE = 255
EXPECTED_CLASSES = 72
LABEL_LEVEL = 1

In [ ]:
import gc
import importlib.metadata
import json
import subprocess
from dataclasses import asdict, replace

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import timm
import torch
from IPython.display import display
from PIL import Image

folders = [Path(REPO_PATH).expanduser()] if REPO_PATH else [Path.cwd(), *Path.cwd().parents]
project_dir = next((p.resolve() for p in folders if (p / "src/train.py").is_file()), None)
if project_dir is None:
    raise FileNotFoundError("ตั้ง REPO_PATH เป็น repo ที่มี src/train.py")
if str(project_dir) not in sys.path:
    sys.path.insert(0, str(project_dir))
from src.audit import DEFAULT_EXTENSIONS, audit_dataset, image_paths
from src.paths import configured_data_dir
from src.split import fingerprint, prepare_split, read_rows
from src.train import TrainConfig, build_model, choose_device, fit, load_split, save_json, make_transform
from src.search import rank_runs


TOTAL_JOBS = 41
if not OWNER.strip() or OWNER == "your_name":
    raise ValueError("กรอก OWNER เป็นชื่อผู้รันก่อน")
if TEAM_SIZE not in (6, 7) or not 1 <= MEMBER_ID <= TEAM_SIZE:
    raise ValueError("ใช้ TEAM_SIZE 6 หรือ 7 และ MEMBER_ID ระหว่าง 1..TEAM_SIZE")
if not isinstance(ROUND, int) or isinstance(ROUND, bool) or ROUND < 1:
    raise ValueError("ROUND ต้องเป็นจำนวนเต็มตั้งแต่ 1 ขึ้นไป")
if JOB_ID is not None and (not isinstance(JOB_ID, int) or isinstance(JOB_ID, bool) or not 1 <= JOB_ID <= TOTAL_JOBS):
    raise ValueError("JOB_ID ต้องเป็น None หรือจำนวนเต็ม 1..41")
if MODE not in {"quick", "full"}:
    raise ValueError("Template ทีมใช้ข้อมูลจริงเท่านั้น: MODE quick หรือ full")

assigned_job_id = JOB_ID if JOB_ID is not None else (ROUND - 1) * TEAM_SIZE + MEMBER_ID
if assigned_job_id > TOTAL_JOBS:
    raise ValueError(f"รอบ {ROUND} สมาชิก {MEMBER_ID} ไม่มีงานแล้ว; งานทั้งหมดมี {TOTAL_JOBS} งาน")

epochs = {"quick": 6, "full": 12}[MODE]
output_base = Path(OUTPUT_ROOT).expanduser().resolve() if OUTPUT_ROOT else project_dir / "results/team_family_search"
results_dir = output_base / f"team-{TEAM_SIZE}" / f"member-{MEMBER_ID:02d}" / f"job-{assigned_job_id:02d}"
results_dir.mkdir(parents=True, exist_ok=True)
assignment_mode = "manual" if JOB_ID is not None else "round"
print("ผู้รัน:", OWNER, "สมาชิก:", MEMBER_ID, "จาก", TEAM_SIZE)
print("รับงาน:", assigned_job_id, "แบบ", assignment_mode, "รอบ", ROUND)
print("torch", torch.__version__, "timm", timm.__version__, "device", choose_device(DEVICE))

## 1. งานที่รับผิดชอบ

Cell ถัดไปแสดงตาราง 41 งานและเลือกเพียงหนึ่งโมเดลตามรอบหรือ `JOB_ID` ที่จองไว้

In [ ]:
team_plan = json.loads((project_dir / "configs/experiments/team_family_assignments.json").read_text())
custom_job = {
    "family": "CustomCNN",
    "model": "custom_cnn",
    "group": "From scratch",
    "tier": "baseline",
    "cost": "low",
    "hypothesis": "baseline CNN ที่อธิบาย convolution/pooling/loss ได้ทุกชั้น",
}
task_table = pd.DataFrame([custom_job, *team_plan["candidates"]])
task_table.insert(0, "job_id", range(1, len(task_table) + 1))
if len(task_table) != TOTAL_JOBS or task_table["family"].duplicated().any():
    raise ValueError("รายการงานต้องมี 41 โมเดลและ family ต้องไม่ซ้ำ")

selected = task_table[task_table.job_id == assigned_job_id].copy()
selected_job = selected.iloc[0].to_dict()
print(f"งาน {assigned_job_id:02d}/41:", selected_job["family"], "->", selected_job["model"])
display(selected)

# ใช้ตารางนี้เช็กงานว่างร่วมกัน แล้วใส่เลขใน JOB_ID หากต้องการหยิบเอง
display(task_table[["job_id", "family", "model", "group", "cost"]])

## 2. ตรวจข้อมูลจริงและ split กลาง

โหลดข้อมูลจาก Drive เฉพาะเมื่อยังไม่มี cache ใช้ภาพจริงทั้งหมดตาม split กลาง
ตรวจข้อมูลกับ hash เดิมก่อนเทรน และแสดงภาพตัวอย่างด้านล่าง ไม่มีการสุ่มสร้าง dataset
หากแจ้งว่า split หาย ให้รับ `train.csv`, `val.csv`, `label_to_index.json`, `split_meta.json` จากผู้ดูแลข้อมูล

In [ ]:
requested_data = Path(DATA_PATH).expanduser() if DATA_PATH else configured_data_dir()
data_dir = (requested_data if requested_data.is_absolute() else project_dir / requested_data).resolve()
requested_split = Path(SPLIT_PATH).expanduser() if SPLIT_PATH else project_dir / "data/splits"
split_dir = (requested_split if requested_split.is_absolute() else project_dir / requested_split).resolve()
expected_classes, label_level = EXPECTED_CLASSES, LABEL_LEVEL
required = ["train.csv", "val.csv", "label_to_index.json", "split_meta.json"]
if not all((split_dir / f).is_file() for f in required):
    raise FileNotFoundError("ต้องรับ split กลางครบ 4 ไฟล์ก่อน: " + str(split_dir))
if (data_dir / ".download_incomplete").exists():
    raise RuntimeError("dataset ยังดาวน์โหลดไม่ครบ")
if not any(image_paths(data_dir, DEFAULT_EXTENSIONS)) and not any(data_dir.glob("*.zip")):
    if not DOWNLOAD_IF_MISSING:
        raise FileNotFoundError("ไม่พบข้อมูลจริง: ตั้ง DATA_PATH หรือเปิด DOWNLOAD_IF_MISSING")
    if data_dir.exists() and any(data_dir.iterdir()):
        raise RuntimeError("โฟลเดอร์มีไฟล์แต่ไม่พบภาพหรือ ZIP: ตรวจ DATA_PATH ก่อนดาวน์โหลด")
    try:
        subprocess.run([sys.executable, "-m", "src.download_data", "--output-dir", str(data_dir)],
                       cwd=project_dir, check=True)
    except BaseException:
        data_dir.mkdir(parents=True, exist_ok=True)
        (data_dir / ".download_incomplete").touch()
        raise

audit_dir = results_dir / MODE / "audit"
audit_report = audit_dataset(data_dir, audit_dir, label_level, DEFAULT_EXTENSIONS)
display(pd.Series({k: audit_report[k] for k in ["valid_images", "class_count", "corrupt_images", "label_errors", "exact_duplicate_groups"]}))
if audit_report["corrupt_images"] or audit_report["label_errors"]:
    raise ValueError("แก้ corrupt images / labels ก่อนทดลอง")
# ใช้ settings ของ split กลาง ไม่ผูก split seed กับ training seed
previous_meta = json.loads((split_dir / "split_meta.json").read_text()) if (split_dir / "split_meta.json").exists() else {}
split_meta = prepare_split(data_dir, audit_dir, split_dir,
    seed=previous_meta.get("seed", 42), val_fraction=previous_meta.get("val_fraction", 0.2),
    expected_classes=expected_classes,
    conflicting_label_policy=previous_meta.get("conflicting_label_policy", "exclude"))
train_rows, val_rows, mapping, _ = load_split(data_dir, split_dir)
if split_meta.get("train_only_classes") or {r["label"] for r in val_rows} != set(mapping):
    print("คำเตือน: คลาสที่ไม่มี validation จะไม่ถูกวัดจากภาพ validation: " + str(split_meta.get("train_only_classes", [])))
counts = pd.crosstab(pd.Series([r["label"] for r in train_rows + val_rows], name="label"),
                     ["train"] * len(train_rows) + ["validation"] * len(val_rows))
display(counts)

print("✅ REAL DATA พร้อมใช้งาน")
print("Dataset root:", data_dir)
print("โฟลเดอร์ภาพจริง:", data_dir / "round2")
print("จำนวนภาพ / คลาส:", audit_report["valid_images"], "/", audit_report["class_count"])

fig, axes = plt.subplots(2, 6, figsize=(12, 5))
sample_rows = pd.DataFrame(train_rows).sort_values(["label", "path"]).groupby("label").head(1).head(12)
for axis, row in zip(axes.flat, sample_rows.itertuples()):
    with Image.open(data_dir / row.path) as image:
        axis.imshow(image.convert("RGB"), cmap="gray")
    axis.set_title(f"label: {row.label}")
for axis in axes.flat:
    axis.axis("off")
plt.suptitle("REAL DATA samples before preprocessing")
plt.tight_layout()
plt.savefig(audit_dir / "dataset_samples.png", dpi=140)
plt.show()

## 3. บันทึกสูตรทดลอง

โมเดล transfer ใช้ pretrained weights, freeze 2 epochs, seed 42, input 224, batch 32,
head LR 0.001, fine-tune LR 0.0001, dropout 0.3 และไม่เปิด augmentation/class weights
Custom CNN ฝึกจากศูนย์ สูตรและ weight tag จริงจะอยู่ใน `protocol.json` และ config ของแต่ละ run
โฟลเดอร์ผลแยกตามทีม/สมาชิก/โหมด/split/สูตร เพื่อไม่เขียนทับงานเพื่อน

In [ ]:
def resolve_candidate(row):
    result = dict(row)
    name = row["model"]
    if name == "custom_cnn":
        return {**result, "architecture": name, "status": "ready", "pretrained_recipe": {}}
    if not timm.is_model(name):
        return {**result, "status": "unavailable", "error": "model absent from installed timm"}
    cfg = timm.models.get_pretrained_cfg(name)
    if cfg is None or not cfg.has_weights:
        return {**result, "status": "unavailable", "error": "no registered pretrained weights"}
    name = name if "." in name or not cfg.tag else f"{name}.{cfg.tag}"
    recipe = cfg.to_dict()
    return {**result, "architecture": name, "status": "ready", "pretrained_recipe": recipe,
            "native_size": list(cfg.input_size), "native_mean": list(cfg.mean), "native_std": list(cfg.std)}

catalog = [resolve_candidate(row) for row in selected.to_dict("records")]
base_config = TrainConfig(seed=SEED, image_size=IMAGE_SIZE, batch_size=BATCH_SIZE, epochs=epochs,
                          freeze_epochs=2, pretrained=True, device=DEVICE, pad_value=PAD_VALUE)
versions = {name: importlib.metadata.version(name) for name in
            ["torch", "torchvision", "timm", "numpy", "scikit-learn", "optuna"]}
contract = {"owner": OWNER, "team_size": TEAM_SIZE, "member_id": MEMBER_ID, "round": ROUND,
            "job_id": assigned_job_id, "assignment_mode": assignment_mode, "mode": MODE,
            "stage": "family_screening", "config": asdict(base_config), "catalog": catalog,
            "split_hash": split_meta["split_hash"], "versions": versions,
            "trainer_hash": fingerprint([(project_dir / "src" / f).read_text()
                                         for f in ["train.py", "split.py", "search.py"]])}
experiment_dir = results_dir / MODE / split_meta["split_hash"][:12] / fingerprint(contract)[:12]
experiment_dir.mkdir(parents=True, exist_ok=True)
save_json(experiment_dir / "protocol.json", contract)
pd.DataFrame(catalog).to_csv(experiment_dir / "candidate_catalog.csv", index=False)
display(pd.DataFrame(catalog).drop(columns=["pretrained_recipe"]))
print("โมเดลของงานนี้:", catalog[0]["family"], "->", catalog[0]["model"])
print("Output:", experiment_dir)

## 4. เทรนโมเดลของงานนี้

Run All หนึ่งครั้งเทรนหนึ่งโมเดล หากล้มเหลวจะบันทึก `complete=false` เพื่อแก้หรือเปิดให้รับงานเดิมใหม่

In [ ]:
template_notebook = json.loads((project_dir / "notebooks/03-team-model-template.ipynb").read_text())
template_source = next(
    cell["source"] for cell in template_notebook["cells"]
    if "screening" in cell.get("metadata", {}).get("tags", [])
)
exec(compile("".join(template_source), "team-template:screening", "exec"))

## 5. ดูภาพจริงกับคำทำนาย

รายงานใช้ validation ชุดเดียวกับการเลือก checkpoint จึงไม่ใช่ unseen test
เมื่อมีผลเดิม ตั้ง `REVIEW_TABLE` แล้วรันส่วนตั้งค่า/imports/ข้อมูลและ cell นี้เพื่อดูผลโดยไม่เทรนใหม่

In [ ]:
template_notebook = json.loads((project_dir / "notebooks/03-team-model-template.ipynb").read_text())
template_source = next(
    cell["source"] for cell in template_notebook["cells"]
    if "visual-screening" in cell.get("metadata", {}).get("tags", [])
)
exec(compile("".join(template_source), "team-template:visual-screening", "exec"))

## 6. ส่งงานและรับงานถัดไป

ส่งทั้งโฟลเดอร์ที่ Cell สุดท้ายพิมพ์ หลังบันทึกสถานะในตารางแชร์แล้วจึงเพิ่ม `ROUND` หรือเลือก `JOB_ID` ใหม่

ผู้รวมผลต้องตรวจงาน 1–41 ไม่ซ้ำ ไม่ขาด และรับเฉพาะงาน `complete=true` ที่มี checkpoint จริง

In [ ]:
template_notebook = json.loads((project_dir / "notebooks/03-team-model-template.ipynb").read_text())
template_source = next(
    cell["source"] for cell in template_notebook["cells"]
    if "handoff" in cell.get("metadata", {}).get("tags", [])
)
exec(compile("".join(template_source), "team-template:handoff", "exec"))